# Urban Mobility Analysis in LATAM

## Objetivo
El objetivo es **evaluar cómo la movilidad urbana se relaciona con la productividad económica en las principales ciudades latinoamericanas**. 

## Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt# importar librerías

In [ ]:
traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv('/datasets/oecd_city_economy.csv')

In [ ]:
traffic.head()

In [ ]:
eco.head()

# Exploración, limpieza y preparación de datos

In [ ]:
traffic.info()
traffic.head(3)

En la estructura del DF traffic, se observa que:
- Las columnas `UpdateTimeUTC` y `UpdateTimeUTCWeekAgo` son de tipo object pero deberian convertirse a tipo datetime para un correcto análisis temporal.
- No se presentan datos ausentes, ya que todas las columnas tienen el mismo número de valores no nulos.

In [ ]:
eco.info()
eco.head(3)

En la estructura del DF eco, se observa que:
- Las columnas `City GDP/capita`, `Unemployment %`, `PM2.5 (µg/m³)` y `Population (M)` son de tipo object, pero representan valores numéricos.
- Estas columnas requieren limpieza previa (eliminación de símbolos como `%` y conversión de formato decimal) antes de ser convertidas a tipo float64.
- No se presentan datos ausentes, ya que todas las columnas tienen valores no nulos.

In [ ]:
# Estandarización de nombres de las columnas de traffic
traffic = traffic.rename(columns={
    "Country": "country",
    "City": "city",
    "UpdateTimeUTC": "update_time_utc",
    "JamsDelay": "jams_delay",
    "TrafficIndexLive": "traffic_index_live",
    "JamsLengthInKms": "jams_length_in_kms",
    "JamsCount": "jams_count",
    "TrafficIndexWeekAgo": "traffic_index_week_ago",
    "UpdateTimeUTCWeekAgo": "update_time_utc_week_ago",
    "TravelTimeLivePer10KmsMins": "travel_time_live_per_10kms_mins",
    "TravelTimeHistoricPer10KmsMins": "travel_time_historic_per_10kms_mins",
    "MinsDelay": "mins_delay"
})
traffic.columns

In [ ]:
# Estandarización de nombres de las columnas de eco
eco = eco.rename (columns={
    'Year':'year', 
    'City':'city', 
    'Country':'country', 
    'City GDP/capita':'city_gdp_capita', 
    'Unemployment %':'unemployment_pct',
    'PM2.5 (μg/m³)':'pm25', 
    'Population (M)':'population_m'
})
eco.columns

In [ ]:
# Conversión las columnas de traffic a tipo fecha con pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors='coerce') 
traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'], errors='coerce')

traffic.info()

In [ ]:
# Limpieza de separadores y conversión columnas numéricas en eco
eco['city_gdp_capita'] = eco['city_gdp_capita'].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)
eco['unemployment_pct'] = eco['unemployment_pct'].astype(str).str.replace('%', '').str.replace(',', '.').astype(float)
eco['population_m'] = eco['population_m'].astype(str).str.replace(',', '.').astype(float)

# Cálculo de la población total en unidades absolutas
eco['population'] = eco['population_m']* 1000000

eco.info()
eco.head(3)

In [ ]:
# Extracción del año de las fechas en update_time_utc
traffic['year'] = traffic['update_time_utc'].dt.year

traffic.head(3)

In [ ]:
# Filtro de registros del año 2024
traffic_2024 = traffic[traffic['year']==2024].copy()
eco_2024 = eco[eco['year']==2024].copy()

display(traffic_2024.head())
display(eco_2024.head())


## EDA

In [ ]:
# Cálculo de  promedios de trafico por ciudad, país y año
traffic_city_year_2024 =  traffic_2024.groupby(['city', 'country', 'year'])\
    ['jams_delay', 'traffic_index_live', 'jams_length_in_kms', 'jams_count', 'mins_delay', 'travel_time_live_per_10kms_mins', 'travel_time_historic_per_10kms_mins']\
    .mean().reset_index()

traffic_city_year_2024.head()

In [ ]:
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

La ciudad con el mayor tiempo promedio de tráfico es la Ciudad de Mexico

In [ ]:
# Selección de columnas clave de tráfico y economía
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_in_kms','jams_count','mins_delay',
             'travel_time_live_per_10kms_mins','travel_time_historic_per_10kms_mins']

right_cols = ['city','year','city_gdp_capita','unemployment_pct','pm25','population']

traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Unión de datasets
merged = pd.merge(traffic_2024_small, eco_2024_small, on=["city", "year"],
    how="inner") # tu código aquí


merged.head()

## Visualización y análisis de relaciones

In [ ]:
# Boxplot para observar el comportamiento de los minutos de congestion JamsDelay
sns.boxplot(x=merged['jams_delay'], showmeans=True)# crea tu gráfico

# promedio para mostrar en título
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.show()


In [ ]:
# Histograma para ver la distribución de la economía (city_gdp_capita)
merged['city_gdp_capita'].hist(bins=10)

plt.title('Distribución del PIB per cápita')
plt.xlabel('PIB per cápita')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# Gráfico de barras para comparar jams_delay y city_gdp_capita por ciudad
merged.plot(x='city', y=['jams_delay', 'city_gdp_capita'], kind='bar')
plt.title('Comparación: Congestión vs PIB por ciudad')
plt.xlabel('Ciudad')
plt.ylabel('Valores')

plt.show()

## Insights Finales

Al comparar el nivel de congestión (jams_delay) con el PIB per cápita por ciudad, no se observa una relación clara entre ambas variables.

Existen ciudades con alto PIB que presentan baja congestión, como Montevideo, así como otras con alto PIB y alta congestión, como Ciudad de México. 

Esto sugiere que el nivel de congestión no depende únicamente del nivel económico, sino que probablemente está influenciado por otros factores como la infraestructura, la densidad poblacional o la planificación urbana.
 

In [ ]:
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)

# 🧾 Resumen ejecutivo
**Contexto & objetivo**
El objetivo del análisis fue evaluar la relación entre la movilidad urbana (medida a través de variables como jams_delay y tiempos de viaje) y la productividad económica (representada por el PIB per cápita). Estas variables son clave para la toma de decisiones, ya que permiten entender si mayores niveles de congestión afectan el desempeño económico de las ciudades o si existen otros factores determinantes.

**Cobertura de datos**
El análisis se realizó sobre datos correspondientes al año 2024, considerando un conjunto de 15 ciudades de distintos países de América Latina. Países como Brasil, Colombia, Argentina, Perú, México, Uruguay y Chile.

**Metodología (alto nivel)**
Se realizó una limpieza de datos que incluyó la estandarización de nombres de columnas, conversión de tipos de datos (especialmente variables numéricas y fechas) y eliminación de columnas irrelevantes. Luego, se agregaron los datos por ciudad–año y se integraron ambos datasets mediante un INNER JOIN utilizando las claves city y year, asegurando trabajar solo con registros coincidentes. Finalmente, se aplicaron validaciones visuales mediante boxplots, histogramas y gráficos comparativos para analizar distribuciones, detectar outliers y observar tendencias generales.

**Hallazgos iniciales**
Los resultados indican que no existe una relación clara o directa entre el PIB per cápita y los niveles de congestión. Se observan casos de ciudades con alto PIB y baja congestión, así como otras con alto PIB y alta congestión, lo que sugiere que la movilidad no depende exclusivamente del nivel económico. Además, se identificaron valores atípicos en congestión, lo que evidencia que ciertas ciudades presentan niveles significativamente más altos que el resto y podrían requerir análisis más detallado.

**Recomendaciones**
Se recomienda priorizar el análisis en ciudades donde coincidan niveles elevados de congestión con indicadores económicos relativamente bajos. En este caso, Bogotá y Lima podrían considerarse candidatas relevantes, ya que muestran niveles de congestión relativamente altos sin un PIB per cápita proporcionalmente elevado. Esto sugiere la necesidad de inversión en infraestructura de transporte y planificación urbana. Asimismo, se recomienda profundizar el análisis incorporando variables adicionales como densidad poblacional, transporte público y expansión urbana, además de validar la calidad y consistencia de las fuentes de datos utilizadas.
